# 03 - Train and evaluate the improved models on your regenerated (v2) data

The tables from notebook 02 use the same column names as the experiment dataset. Point `PROC` at them and
the experiment scripts run unchanged. The extra v2 drivers (ERA5-Land, SMAP, API, LST, ...) are added to the
feature list automatically.

**Recommended reporting for the revised paper**
* Leave-stations-out and leave-one-region-out evaluation, not only random 80/20.
* Pooled R², RMSE and ubRMSE, plus median per-station r and ubRMSE.
* A persistence baseline whenever an in-situ lag is used.
* A satellite-only model (no in-situ input) as the main product.

In [ ]:
import sys; sys.path.insert(0, '../src'); sys.path.insert(0, '../gee')
import common
common.PROC = '../data/processed_v2'          # <- tables from assemble_v2.py
from assemble_v2 import V2_EXTRA_FEATS
import pandas as pd
a = pd.read_parquet(f'{common.PROC}/anchor.parquet')
extra = [c for c in V2_EXTRA_FEATS if c in a and a[c].notna().mean() > 0.5]
common.IMPROVED_SAT[:] = common.IMPROVED_SAT + [c for c in extra if c not in common.IMPROVED_SAT]
common.IMPROVED_LAG[:] = common.IMPROVED_SAT + ['SM_lag1d']
print(len(common.IMPROVED_SAT), 'satellite-only features')

## Anchor (Sentinel-1 date) models: protocols, persistence, trees vs TempoSAR vs hybrid, transfer learning

In [ ]:
import exp1_baselines, exp2_transfer as e2
exp1_baselines.main()
lab, full = e2.load_for_deep()
e2.part_a(lab, full)
e2.part_b(lab, full, regions=[r for r, n in lab.groupby('region').site.nunique().items() if n >= 5])

## Daily (gap-fill) models with daily drivers
LGBM and a DailyLSTM are compared. The DailyLSTM is pretrained on ERA5-Land / SMAP soil moisture and then fine-tuned on in-situ data.

In [ ]:
!cd ../src && python daily_v2.py --daily ../data/processed_v2/daily.parquet --out ../results/v2 --epochs 15